In [1]:
import sys

if sys.platform == "win32":
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_community.utilities import SQLDatabase
from langchain.agents import AgentState
from langchain_tavily import TavilySearch
from langchain.tools import tool
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage, AIMessage
from langgraph.types import Command
from langchain.agents import create_agent
from dotenv import load_dotenv

load_dotenv()

True

# Multi Agent

Multi Agent，既多智能体协作系统，专门用来处理复杂的任务流程。


## 1.概述

通常在以下几种情况下我们会使用Multi Agent：
- **上下文管理(Context Management)**：如果同时需要调用的工具很多，或者上下文内容很多，我们可以将任务拆分，交给不同的Agent处理
- **分布式开发(Distributed development)**：不同的团队独立开发和维护自己的Agent，并将他们组合成一个更大的Agent
- **并行(Parallelization)**：将任务拆分为多个子任务，并交给专门的Agent处理，并同时执行它们以加快处理速度

多智能体协作的模式有很多种，比较常见的有：
- **Subagents**：**子代理模式**，一个主Agent将多个子Agent作为Tool来协调使用。所有请求都由主Agent处理，决定何时以及如何调用每个子Agent
- **Handoffs**：**传递模型**，随着任务的执行改变state中的任务状态，从而触发路由变更，切换到其它Agent。因此每个Agent都可以与用户交互，处理用户请求并返回响应
- **Skills**：**技能模式**，只有1个Agent，根据任务按需加载Skill或知识
- **Router**：**路由模式**，1个负责路由的Agent对用户请求进行分类，将请求导向给一个或多个专门的Agent。最后再由一个Agent负责总结结果。


## 2.案例

接下来，我们以Subagents这种模式为例，来学习多智能体协作的模式。

我们计划开发一个婚礼策划智能体，Mary AI，它将包含以下三个子Agent：
- 旅行Agent：负责为你前往婚礼目的地寻找往返机票
- 场地Agent：负责在根据宾客人数在网上搜索合适的婚礼场地
- DJ Agent：负责在音乐数据库筛选符合用户需求的歌单，并计算出预算

最后，我们还会定义一个主Agent，负责协调工作，以及生成最终的婚礼计划方案。

### 2.1.Subagent 旅行

接下来是旅行Agent，需要查询航班，所以我们要使用Kiwi这个MCP，还有时间查询MCP

In [2]:
# MCP客户端，包含Time、Kiwi两个MCP
client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "http",
                "url": "https://mcp.kiwi.com"
        },
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=Asia/Shanghai"
            ]
        }
    }
)

tools = await client.get_tools()

In [3]:
# Travel agent
travel_agent = create_agent(
    model="deepseek-flash",
    tools=tools,
    system_prompt="""
    你是一个旅行助手Agent。你为用户搜索到达婚礼地点的理想航班，您不能再追问任何后续问题。
    您必须根据以下标准找到最佳航班选择：
    - 价格（最低，经济舱）
    - 持续时间（最短）
    - 日期（您认为在该地点举行婚礼的最佳时间）
    为了使事情变得简单，只寻找一张票，单程。
    你不会得到额外的信息，只有出发地和目的地。你的工作是批判性地思考最佳选择。
    一旦你找到了最好的选项，让用户知道你的候选列表。
    调用搜索航班工具时，如果是地点在中国，需要使用zh-cn作为Locale参数。
    """
)

In [4]:
# 测试
response = await travel_agent.ainvoke(
    {"messages": [HumanMessage(content="给我订一张明天从北京直飞上海的机票。")]},
)
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

给我订一张明天从北京直飞上海的机票。
================================== Ai Message ==================================
Tool Calls:
  get_current_time (call_00_MrMKaLLSQbN9v08ujCOF8697)
 Call ID: call_00_MrMKaLLSQbN9v08ujCOF8697
  Args:
    timezone: Asia/Shanghai
================================= Tool Message =================================
Name: get_current_time

[{'type': 'text', 'text': '{\n  "timezone": "Asia/Shanghai",\n  "datetime": "2026-09-19T17:03:11+08:00",\n  "day_of_week": "Saturday",\n  "is_dst": false\n}', 'id': 'lc_b9cc25e2-d166-44ba-97fe-4f73d5512727'}]
================================== Ai Message ==================================
Tool Calls:
  search-flight (call_00_bttSFM3VKw1laLdvqGdZ6341)
 Call ID: call_00_bttSFM3VKw1laLdvqGdZ6341
  Args:
    flyFrom: 北京
    flyTo: 上海
    departureDate: 20/09/2026
    cabinClass: M
    max_sector_stopovers: 0
    locale: zh-cn
    sort: price
========================

### 2.2.Subagent 场馆

场馆我们直接使用Tavily工具，在网上搜索合适的场馆。


In [5]:
# 定义Tavily web_search工具
web_search = TavilySearch(
    max_results=5,
    topic="general",
)

In [6]:
# 创建 Venue agent
venue_agent = create_agent(
    model="deepseek-flash",
    tools=[web_search],
    system_prompt="""
    你是场地专家，你在指定的位置搜索满足要求的场地。
    你不能再追问任何后续问题，你必须根据以下标准找到最佳的场地选择：
    - 价格（最低）
    - 容量（完全匹配）
    - 评价（最高）
    您可能需要进行多次搜索以迭代地找到最佳选项。
    """
)

In [7]:
response = venue_agent.invoke(
    {"messages": [HumanMessage("上海有哪些适合举办婚礼的场馆？100人左右。")]}
)
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

上海有哪些适合举办婚礼的场馆？100人左右。
================================== Ai Message ==================================

I'll help you find wedding venues in Shanghai suitable for around 100 people. Let me search for options.
Tool Calls:
  tavily_search (call_00_h2LCN4r0316LTyow269y3764)
 Call ID: call_00_h2LCN4r0316LTyow269y3764
  Args:
    query: 上海 婚礼场馆 100人 婚宴场地 推荐
    search_depth: advanced
  tavily_search (call_01_rGGqV9bLjWBMOH8S8h0z6509)
 Call ID: call_01_rGGqV9bLjWBMOH8S8h0z6509
  Args:
    query: 上海 婚礼会所 小型婚礼 10桌 场地
    search_depth: advanced
================================= Tool Message =================================
Name: tavily_search

{"query": "上海 婚礼场馆 100人 婚宴场地 推荐", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.hunliji.com/bai_ke/detail_82856", "title": "上海办婚宴最多的酒店前10名（2025新）【婚礼纪】", "content": "### 莉维娜婚礼艺术中心（世纪公园店、陆家嘴店、歇浦路店）\n\n三家门店评分均为 5 分。其中，世纪公园店有 5 个婚宴

### 2.3.Subagent 歌单

最后是歌单Agent，我已经提供好了一些模拟数据，保存在Sqlite文件Chinook.db中。

我们需要定义一个能查询数据库的Tool，方便Agent查询歌单数据。


In [8]:
db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def query_playlist_db(query: str) -> str:

    """Query the database for playlist information"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database: {e}"

In [10]:
# Playlist agent
playlist_agent = create_agent(
    model="deepseek-flash",
    tools=[query_playlist_db],
    system_prompt="""
    你是播放列表专家。查询sql数据库并为给定类型的婚礼策划完美的播放列表。
    一旦你有了你的播放列表，计算播放列表的总时长和成本，每首歌都有一个相关的价格。
    如果在查询数据库时遇到错误，请尝试通过更改查询来修复它们。
    不要空手而归，继续尝试查询数据库，直到你找到一个歌曲列表。
    您可能需要进行多个查询以迭代地找到最佳选项。
    """
)

In [11]:
response = playlist_agent.invoke(
    {"messages": [HumanMessage("爵士风格")]}
)
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

爵士风格
================================== Ai Message ==================================

I'll help you create a jazz-style wedding playlist. Let me start by exploring the database.
Tool Calls:
  query_playlist_db (call_00_AhbqDqBkUu1OtZUWm84u2786)
 Call ID: call_00_AhbqDqBkUu1OtZUWm84u2786
  Args:
    query: SELECT * FROM songs LIMIT 10
================================= Tool Message =================================
Name: query_playlist_db

Error querying database: (sqlite3.OperationalError) no such table: songs
[SQL: SELECT * FROM songs LIMIT 10]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
================================== Ai Message ==================================

Let me find the correct table names.
Tool Calls:
  query_playlist_db (call_00_F9kwK408DGJIgV6yGZEj1184)
 Call ID: call_00_F9kwK408DGJIgV6yGZEj1184
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table'
=======

### 2.4.主Agent（Coordinator）

最后，就是主Agent了，也是整个Agent的核心协调者。它负责与用户交互，收集婚礼信息，调用各个Subagent，筹备婚礼计划。

#### 2.4.1.定义state
首先，我们需要定义一个state，记录婚礼风格有关的信息，包括：
- 婚礼人数
- 音乐风格
- 出发地
- 婚礼举办地


In [12]:
class WeddingState(AgentState):
    origin: str         # 起点
    destination: str    # 目的地
    guest_count: str    # 宾客数量
    genre: str          # 婚礼风格


#### 2.4.2.Tools


首先，要执行婚礼规划任务必须知道婚礼信息，也就是state中要记录的数据。AI通过与用户沟通可以获得这些信息，但是必须有一个tool负责更新state信息，当AI通过与用户聊天获取这些信息后，调用该tool更新到state中。

In [13]:
@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str, runtime: ToolRuntime) -> str:
    """当你询问用户，知道下列所有值时更新State：origin、destination、guest_count、genre"""
    return Command(update={
        "origin": origin,
        "destination": destination,
        "guest_count": guest_count,
        "genre": genre,
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]}
        )

按照Subagent模式，主Agent需要把Subagents当做一个个Tool来协调使用。所以，我们接下来就先定义3个Tool，分别对应3个Subagent.

In [14]:
@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """旅行Agent搜索飞往理想婚礼地点的航班."""
    origin = runtime.state["origin"]
    destination = runtime.state["destination"]

    content = f"查询从{origin}到{destination}的航班"
    response = await travel_agent.ainvoke({"messages": [HumanMessage(content)]})
    return response['messages'][-1].content

@tool
def search_venues(runtime: ToolRuntime) -> str:
    """场地Agent根据给定的地点和宾客数量寻找最佳的婚礼场地."""
    destination = runtime.state["destination"]
    capacity = runtime.state["guest_count"]

    content = f"在{destination}寻找适合{capacity}宾客的婚礼场地"
    response = venue_agent.invoke({"messages": [HumanMessage(content)]})
    return response['messages'][-1].content

@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """播放列表Agent根据类型风格策划完美的播放列表."""
    genre = runtime.state["genre"]

    content = f"策划符合{genre}风格的婚礼播放列表的曲目"
    response = playlist_agent.invoke({"messages": [HumanMessage(content)]})
    return response['messages'][-1].content


#### 2.4.3.Agent

接下来，就是创建主Agent了，把刚刚的4个Tools都注册给它，并给它设定核心协调者的任务。

In [15]:
from langchain.agents import create_agent

coordinator = create_agent(
    model="deepseek-flash",
    tools=[search_flights, search_venues, suggest_playlist, update_state],
    state_schema=WeddingState,
    system_prompt="""
    你是婚礼协调员。你负责与用户沟通，找到更新状态所需的所有信息，并更新到state。
    一旦完成，你就可以委派任务了。将查找航班、搜索场地和策划播放列表的任务委派给您的专家Agent。
    当你收到他们的答案，为我设计一场完美的婚礼方案。
    """
)

#### 2.4.4.测试

In [16]:
response = await coordinator.ainvoke(
    {
        "messages": [HumanMessage(content="我来自北京，我想在上海举办一场100人的婚礼，爵士风格的。")],
    }
)

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

我来自北京，我想在上海举办一场100人的婚礼，爵士风格的。
================================== Ai Message ==================================

I'll update the state with all the information you've provided.
Tool Calls:
  update_state (call_00_FRomBkDrLGMBHuO7FXAH1176)
 Call ID: call_00_FRomBkDrLGMBHuO7FXAH1176
  Args:
    origin: 北京
    destination: 上海
    guest_count: 100
    genre: 爵士
================================= Tool Message =================================
Name: update_state

Successfully updated state
================================== Ai Message ==================================

Now I have everything I need. Let me delegate the tasks to my expert agents.
Tool Calls:
  search_flights (call_00_W5s7xLjQumsToEMbAIUP5246)
 Call ID: call_00_W5s7xLjQumsToEMbAIUP5246
  Args:
  search_venues (call_01_vbjMF7lfYZEpZ6x8L2JO2856)
 Call ID: call_01_vbjMF7lfYZEpZ6x8L2JO2856
  Args:
  suggest_playlist (call_02_mIqMGKYnG8CYDG1ocG3x9637)
 